# TFT (Temporal Fusion Transformer) - Deep Learning Time Series Model
imports, set up wandb.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import wandb

from neuralforecast import NeuralForecast
from neuralforecast.models import TFT

from preprocessing import load_raw, build_features, to_long_format, wmae, MARKDOWN_COLS
from src.config import WANDB_ENTITY, WANDB_PROJECT
from src.wandb_utils import safe_wandb_init, save_and_log_pipeline_artifact

np.random.seed(42)
print("W&B entity:", WANDB_ENTITY)
print("W&B project:", WANDB_PROJECT)

W&B entity: gchal22-free-university-of-tbilisi-
W&B project: store_sales_forecast


## 1. Load & Reshape Data for neuralforecast

In [2]:
FUTR_EXOG_COLS = ["IsHoliday", "CPI", "Unemployment", "Fuel_Price", "Temperature"] + MARKDOWN_COLS

train, test, features, stores = load_raw()
train_full = build_features(train, features, stores)
long_df = to_long_format(train_full, extra_cols=FUTR_EXOG_COLS)

print(long_df.shape)
long_df.head()

(421570, 13)


,unique_id,ds,y,IsHoliday,CPI,Unemployment,Fuel_Price,Temperature,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5
0,10_1,2010-02-05,40212.84,False,126.442065,9.765,2.962,54.34,0.0,0.0,0.0,0.0,0.0
1,10_1,2010-02-12,67699.32,True,126.496258,9.765,2.828,49.96,0.0,0.0,0.0,0.0,0.0
2,10_1,2010-02-19,49748.33,False,126.526286,9.765,2.915,58.22,0.0,0.0,0.0,0.0,0.0
3,10_1,2010-02-26,33601.22,False,126.552286,9.765,2.825,52.77,0.0,0.0,0.0,0.0,0.0
4,10_1,2010-03-05,36572.44,False,126.578286,9.765,2.877,55.92,0.0,0.0,0.0,0.0,0.0


## 2. Filter Series

In [3]:
H = 39
INPUT_SIZE = 52
MIN_LEN = INPUT_SIZE + H

long_df_all = long_df.copy()  # unfiltered; Section 4's final fit refilters against INPUT_SIZE only
series_len = long_df.groupby("unique_id").size()
keep_ids = series_len[series_len >= MIN_LEN].index

print(f"series total: {series_len.shape[0]}")
print(f"series kept (len >= {MIN_LEN}): {len(keep_ids)}")
print(f"series dropped (too short): {series_len.shape[0] - len(keep_ids)}")

long_df = long_df[long_df["unique_id"].isin(keep_ids)].reset_index(drop=True)
print("rows after filtering:", long_df.shape[0])

with safe_wandb_init(
    run_name="TFT_Preprocessing",
    group="TFT",
    job_type="preprocessing",
    config={
        "horizon_h": H,
        "input_size": INPUT_SIZE,
        "min_series_length": MIN_LEN,
        "futr_exog_cols": FUTR_EXOG_COLS,
    },
) as run:
    wandb.log({
        "series_total": series_len.shape[0],
        "series_kept": len(keep_ids),
        "series_dropped": series_len.shape[0] - len(keep_ids),
        "rows_kept": long_df.shape[0],
    })

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


series total: 3331
series kept (len >= 91): 2900
series dropped (too short): 431
rows after filtering: 410069


wandb: Currently logged in as: abeku20 (abeku20-free-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\wandb\run-20260712_170241-1icpw90d
wandb: Run `wandb offline` to turn off syncing.


...

wandb: uploading summary


wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml


wandb: 
wandb: Run history:
wandb:      rows_kept ▁
wandb: series_dropped ▁
wandb:    series_kept ▁
wandb:   series_total ▁
wandb: 
wandb: Run summary:
wandb:      rows_kept 410069
wandb: series_dropped 431
wandb:    series_kept 2900
wandb:   series_total 3331
wandb: 


wandb:  View run TFT_Preprocessing at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast/runs/1icpw90d
wandb:  View project at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260712_170241-1icpw90d\logs


## 3. Time-Based Holdout Cross-Validation

In [4]:
MAX_STEPS = 300  # kept small for a fast scaffold run; raise for real training

model = TFT(
    h=H,
    input_size=INPUT_SIZE,
    futr_exog_list=FUTR_EXOG_COLS,
    max_steps=MAX_STEPS,
    hidden_size=32,
    n_head=2,
    scaler_type="robust",
    random_seed=42,
    enable_progress_bar=False,
    logger=False,
)
nf = NeuralForecast(models=[model], freq="W-FRI")

cv_df = nf.cross_validation(df=long_df, n_windows=None, test_size=H)
cv_df = cv_df.merge(
    long_df[["unique_id", "ds", "IsHoliday"]], on=["unique_id", "ds"], how="left"
)
cv_df.head()

Seed set to 42


GPU available: False, used: False


TPU available: False, using: 0 TPU cores



  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | padder_train            | ConstantPad1d            | 0      | train
2 | scaler                  | TemporalNorm             | 0      | train
3 | embedding               | TFTEmbedding             | 704    | train
4 | temporal_encoder        | TemporalCovariateEncoder | 143 K  | train
5 | temporal_fusion_decoder | TemporalFusionDecoder    | 17.0 K | train
6 | output_adapter          | Linear                   | 33     | train
-----------------------------------------------------------------------------
161 K     Trainable params
0         Non-trainable params
161 K     Total params
0.645     Total estimated model params size (MB)
232       Modules in train mode
0         Modules in eval mode


C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


`Trainer.fit` stopped: `max_steps=300` reached.


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


,unique_id,ds,cutoff,TFT,y,IsHoliday
0,10_1,2012-02-03,2012-01-27,32915.199219,36444.00,False
1,10_1,2012-02-10,2012-01-27,35136.851562,50434.11,True
2,10_1,2012-02-17,2012-01-27,29376.574219,74930.33,False
3,10_1,2012-02-24,2012-01-27,25940.812500,28751.57,False
4,10_1,2012-03-02,2012-01-27,27647.177734,30525.88,False


In [5]:
val_wmae = wmae(cv_df["y"], cv_df["TFT"], cv_df["IsHoliday"])
val_mae = (cv_df["y"] - cv_df["TFT"]).abs().mean()

print(f"Validation WMAE: {val_wmae:.2f}")
print(f"Validation MAE:  {val_mae:.2f}")

with safe_wandb_init(
    run_name="TFT_CV",
    group="TFT",
    job_type="cross_validation",
    config={
        "horizon_h": H,
        "input_size": INPUT_SIZE,
        "max_steps": MAX_STEPS,
        "scaler_type": "robust",
        "futr_exog_cols": FUTR_EXOG_COLS,
    },
) as run:
    wandb.log({"val_wmae": val_wmae, "val_mae": val_mae})

Validation WMAE: 2593.39
Validation MAE:  2565.06


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\wandb\run-20260712_171131-1j6ge3vb
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run TFT_CV


wandb:  View project at https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast


...

wandb: uploading summary


wandb: uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json


wandb: 
wandb: Run history:
wandb:  val_mae ▁
wandb: val_wmae ▁
wandb: 
wandb: Run summary:
wandb:  val_mae 2565.05676
wandb: val_wmae 2593.39071
wandb: 


wandb:  View run TFT_CV at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast/runs/1j6ge3vb
wandb:  View project at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260712_171131-1j6ge3vb\logs


## 4. Final Model Training (Full History) & Pipeline Export

In [6]:
final_keep_ids = series_len[series_len >= INPUT_SIZE].index
long_df_final = long_df_all[long_df_all["unique_id"].isin(final_keep_ids)].reset_index(drop=True)
print(f"series kept for final fit (len >= {INPUT_SIZE}): {len(final_keep_ids)} (vs {len(keep_ids)} for CV)")

final_model = TFT(
    h=H,
    input_size=INPUT_SIZE,
    futr_exog_list=FUTR_EXOG_COLS,
    max_steps=MAX_STEPS,
    hidden_size=32,
    n_head=2,
    scaler_type="robust",
    random_seed=42,
    enable_progress_bar=False,
    logger=False,
)
nf_final = NeuralForecast(models=[final_model], freq="W-FRI")
nf_final.fit(df=long_df_final)

SAVE_PATH = "models/tft_nf"
nf_final.save(path=SAVE_PATH, overwrite=True, save_dataset=True)
print("saved to", SAVE_PATH)

Seed set to 42


series kept for final fit (len >= 52): 2991 (vs 2900 for CV)


GPU available: False, used: False


TPU available: False, using: 0 TPU cores



  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | padder_train            | ConstantPad1d            | 0      | train
2 | scaler                  | TemporalNorm             | 0      | train
3 | embedding               | TFTEmbedding             | 704    | train
4 | temporal_encoder        | TemporalCovariateEncoder | 143 K  | train
5 | temporal_fusion_decoder | TemporalFusionDecoder    | 17.0 K | train
6 | output_adapter          | Linear                   | 33     | train
-----------------------------------------------------------------------------
161 K     Trainable params
0         Non-trainable params
161 K     Total params
0.645     Total estimated model params size (MB)
232       Modules in train mode
0         Modules in eval mode


C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


`Trainer.fit` stopped: `max_steps=300` reached.


saved to models/tft_nf


In [7]:
from src.dl_models import TFTForecastPipeline

with safe_wandb_init(
    run_name="TFT_Final",
    group="TFT",
    job_type="final_training",
    config={
        "horizon_h": H,
        "input_size": INPUT_SIZE,
        "max_steps": MAX_STEPS,
        "scaler_type": "robust",
    },
) as run:
    wandb.log({"val_wmae": val_wmae, "final_series_kept": len(final_keep_ids)})

    save_and_log_pipeline_artifact(
        run,
        "tft-pipeline",
        dirs={"nf_model": SAVE_PATH},
        files={
            "features.csv": "data/raw/features.csv",
            "stores.csv": "data/raw/stores.csv",
        },
        metadata={
            "architecture": "TFT",
            "metric": "WMAE",
            "val_wmae": val_wmae,
            "horizon_h": H,
            "input_size": INPUT_SIZE,
            "max_steps": MAX_STEPS,
            "uses_raw_test": True,
            "notes": "Only DL model using future-known exogenous variables. predict() builds "
                     "futr_df from nf.make_future_dataframe() (not test.csv's own dates -- the "
                     "raw dataset has real gaps in some series' weekly history) and sources "
                     "exogenous values from the bundled features.csv/stores.csv. Load via "
                     "src.dl_models.TFTForecastPipeline.load(nf_path, features_path, stores_path).",
        },
        aliases=["latest", "candidate"],
    )
    print("Logged pipeline to W&B run:", run.id)

wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\wandb\run-20260712_172204-58h1d5nn
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run TFT_Final


wandb:  View project at https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast


wandb:  View run at https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast/runs/58h1d5nn


...

wandb: uploading artifact tft-pipeline; uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml


wandb: uploading artifact tft-pipeline


wandb: 
wandb: Run history:
wandb: final_series_kept ▁
wandb:          val_wmae ▁
wandb: 
wandb: Run summary:
wandb: final_series_kept 2991
wandb:          val_wmae 2593.39071
wandb: 


wandb:  View run TFT_Final at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast/runs/58h1d5nn
wandb:  View project at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast
wandb: Synced 4 W&B file(s), 0 media file(s), 7 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260712_172204-58h1d5nn\logs
